In [10]:
import torch
from torch import nn
import math
import copy
import numpy as np

In [6]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [58]:
def attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        print("-------scores-------\n", scores)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, v)

class MultiheadAttention(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        assert demb % h == 0
        self.d_k = demb // h
        self.h = h
        self.demb = demb
        self.linears = clones(nn.Linear(demb, demb), 4)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1) #adds a dimension to mask at second position for head axis
        nbatches = query.size(0)

        query, key, value = [linear(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) for linear, x in zip(self.linears, (query, key, value))]
        """ 
            this results in query, key, value of shape (batch_size, num_heads, seq_len, d_k) 
            taking dv=dk=demb//h
        """

        """
            contiguous function maintains continuty in memory space even after transpose, or else it might show an error with view function is used to reshape the tensor to the desired shape directly from memory
        """
        x = attention(query, key, value, mask=mask, dropout=self.dropout) #type:ignore
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.demb)
        return self.linears[-1](x)

In [53]:
np.random.seed(42)

x = np.random.randn(3, 4)

mha = MultiheadAttention(demb=4, h=2)

X = torch.tensor(x, dtype=torch.float32).unsqueeze(0)

print(X)  # (1, 3, 4)

mask = np.tril(np.ones((3, 3)))
mask = torch.tensor(mask, dtype=torch.float32).view(1, 3, 3)

print(mask.shape)

out = mha.forward(X, X, X, mask)
print(out.shape)

out

tensor([[[ 0.4967, -0.1383,  0.6477,  1.5230],
         [-0.2342, -0.2341,  1.5792,  0.7674],
         [-0.4695,  0.5426, -0.4634, -0.4657]]])
torch.Size([1, 3, 3])
--------scores-------
 tensor([[[[ 0.7427,    -inf,    -inf],
          [ 0.9622,  1.2613,    -inf],
          [ 0.2227,  0.2630,  0.1114]],

         [[ 0.3715,    -inf,    -inf],
          [ 0.3715,  0.6278,    -inf],
          [-0.0663, -0.0738,  0.0121]]]], grad_fn=<MaskedFillBackward0>)
torch.Size([1, 3, 4])


tensor([[[ 0.4618, -0.9911,  0.0964, -0.2825],
         [ 0.3299, -0.9510,  0.3596, -0.4410],
         [ 0.3480, -0.8339,  0.1572, -0.4410]]], grad_fn=<ViewBackward0>)

In [60]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

In [ ]:
import torch.nn.functional as F

"""
    SwiGLU(x, W, V) = [(xW + b) ⊙ σ(xW + b)] ⊙ (xV + c)
    FFN_SwiGLU(x) = SwiGLU(x, W, V) · W_out + b_out

    W, V : demb x dff
    W_out : dff x demb
    b,c : dff
    b_out : demb
    ⊙ - element-wise multiplication (Hadamard product)
"""

class SwiGLU(nn.Module):
    def __init__(self, demb):
        super().__init__()
        dff = demb * 8/3
        self.w_up = nn.Linear(demb, dff)
        self.w_gate = nn.Linear(demb, dff)
        self.w_down = nn.Linear(dff, demb)

        # often kaiming works best in cases of swish or silu, though we are using sigmoid so we have to experiment to choose betweeen kaiming and xavier initialization

        nn.init.kaiming_uniform_(self.w_gate.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_up.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_down.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.w_down(F.silu(self.w_up(x)) * self.w_gate(x))

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        self.attn = MultiheadAttention(demb, h, dropout)
        """
            the older tranformers used ff networks with linear layers and ReLU activation, but newer ones use SwiGLU which is a gated activation function that can capture more complex relationships in the data
        """
        self.dropout = nn.Dropout(p=dropout)
        self.norm = nn.RMSNorm(demb)

    def forward(self, x, mask=None):
        attn_output = self.attn.forward(x, x, x, mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm(x)

        ff = SwiGLU(x.size(-1))
        ff_output = ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm(x)

        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = clones(EncoderBlock(config.demb, config.h, config.dropout), config.N)
        self.norm = nn.RMSNorm(config.demb)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer.forward(x, mask=mask)
        return self.norm(x)

In [56]:
class DecoderBlock(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(demb, h, dropout)
        self.cross_attn = MultiheadAttention(demb, h, dropout)
        self.dropout = nn.Dropout(p=dropout)
        self.norm = nn.RMSNorm(demb)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        self_attn_output = self.self_attn.forward(x, x, x, mask=tgt_mask)
        x = x + self.dropout(self_attn_output)
        x = self.norm(x)

        cross_attn_output = self.cross_attn.forward(x, enc_output, enc_output, mask=src_mask)
        x = x + self.dropout(cross_attn_output)
        x = self.norm(x)

        ff = SwiGLU(x.size(-1))
        ff_output = ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm(x)

        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = clones(DecoderBlock(config.demb, config.h, config.dropout), config.N)
        self.norm = nn.RMSNorm(config.demb)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer.forward(x, enc_output, src_mask=src_mask, tgt_mask=tgt_mask)
        return self.norm(x)

In [64]:
from pathlib import Path
import urllib.request
import gzip
import shutil

BASE = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw"
FILES = [
    "train.de.gz", "train.en.gz",
    "val.de.gz", "val.en.gz",
    "test_2016_flickr.de.gz", "test_2016_flickr.en.gz",
]

out = Path("multi30k_de_en")
out.mkdir(exist_ok=True)

for name in FILES:
    gz_path = out / name
    txt_path = out / name.replace("test_2016_flickr", "test").replace(".gz", "")
    url = f"{BASE}/{name}"
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, "rb") as f_in, open(txt_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Done. Files created:")
for p in sorted(out.glob("*.de")) + sorted(out.glob("*.en")):
    print(p)


Done. Files created:
multi30k_de_en\test.de
multi30k_de_en\train.de
multi30k_de_en\val.de
multi30k_de_en\test.en
multi30k_de_en\train.en
multi30k_de_en\val.en


In [65]:
import torch

from torchtext.data import Field, Example, Dataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


SRC = Field(
    tokenize="spacy",
    tokenizer_language="de_core_news_sm",
    init_token="<sos>",
    eos_token="<eos>",
    lower=True
)

TRG = Field(
    tokenize="spacy",
    tokenizer_language="en_core_web_sm",
    init_token="<sos>",
    eos_token="<eos>",
    lower=True
)


def load_translation_dataset(src_path, trg_path, src_field, trg_field):
    fields = [("src", src_field), ("trg", trg_field)]
    examples = []

    with open(src_path, encoding="utf-8") as src_file, open(trg_path, encoding="utf-8") as trg_file:
        for src_line, trg_line in zip(src_file, trg_file):
            src_line = src_line.strip()
            trg_line = trg_line.strip()

            if src_line and trg_line:
                example = Example.fromlist(
                    [src_line, trg_line],
                    fields
                )
                examples.append(example)

    return Dataset(examples, fields)


train_data = load_translation_dataset(
    "multi30k_de_en/train.de",
    "multi30k_de_en/train.en",
    SRC,
    TRG
)

valid_data = load_translation_dataset(
    "multi30k_de_en/val.de",
    "multi30k_de_en/val.en",
    SRC,
    TRG
)

test_data = load_translation_dataset(
    "multi30k_de_en/test.de",
    "multi30k_de_en/test.en",
    SRC,
    TRG
)

In [ ]:
print(vars(train_data.examples[1])) #type:ignore

{'src': ['mehrere', 'männer', 'mit', 'schutzhelmen', 'bedienen', 'ein', 'antriebsradsystem', '.'], 'trg': ['several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']}


In [ ]:
from dataclasses import dataclass

@dataclass
class config:
    demb: int = 512
    h: int = 8
    N: int = 6
    dropout: float = 0.1



# def make_model(
#     src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1
# ):
#     "Helper: Construct a model from hyperparameters."
#     c = copy.deepcopy
#     attn = MultiHeadedAttention(h, d_model)
#     ff = PositionwiseFeedForward(d_model, d_ff, dropout)
#     position = PositionalEncoding(d_model, dropout)
#     model = EncoderDecoder(
#         Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
#         Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
#         nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
#         nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
#         Generator(d_model, tgt_vocab),
# #     )

#     # This was important from their code.
#     # Initialize parameters with Glorot / fan_avg.
#     for p in model.parameters():
#         if p.dim() > 1:
#             nn.init.xavier_uniform_(p)
#     return model
